In [1]:
# [Cell 1] V28.1: 修正 Lambda 形狀推斷錯誤
import h5py
import numpy as np
import keras
import keras.layers as L
from keras.models import Model
from collections import defaultdict

def get_transunet_v28_fix(input_shape=(128, 128, 128, 1)):
    inputs = L.Input(shape=input_shape)
    
    # --- Encoder ---
    x = L.Conv3D(64, 7, padding="same", name="stem_conv", use_bias=False)(inputs)
    x = L.BatchNormalization(name="batch_normalization")(x)
    x = L.Activation("relu")(x)
    stem = x
    
    p1 = L.MaxPooling3D(2)(x)
    
    x = L.Conv3D(128, 1, padding="same", name="bottleneck_proj", use_bias=False)(p1)
    x = L.Activation("relu")(x)
    
    # Squeeze
    x_slice = L.Lambda(lambda t: t[..., :4], output_shape=(None, None, None, 4))(x) # 顯式指定輸出形狀作為保險
    
    x = L.Conv3D(128, 3, padding="same", name="conv3d", use_bias=False)(x_slice)
    x = L.Activation("relu")(x)
    
    x = L.Conv3D(256, 1, padding="same", name="conv3d_1", use_bias=False)(x)
    x = L.Activation("relu")(x)
    
    # --- Decoder ---
    u1 = L.UpSampling3D(2)(x)
    x = L.Concatenate()([u1, stem]) # 320ch
    
    x = L.Conv3D(32, 3, padding="same", name="decode_block_1", use_bias=False)(x)
    x = L.Activation("relu")(x)
    
    x = L.Concatenate()([x, stem]) # 96ch
    
    x = L.Conv3D(16, 3, padding="same", name="decode_block_2", use_bias=False)(x)
    x = L.Activation("relu")(x)
    
    # --- Output Head (16 -> 2) ---
    x = L.Conv3D(2, 1, padding="same", name="output_head", use_bias=False)(x)
    
    # 關鍵修正：拆解 Lambda 為標準 Keras 層以避免推斷錯誤
    # 1. 分離通道 0 (Background) 和 通道 1 (Ink)
    # 使用切片層或 Lambda 帶形狀
    # 在 Keras Functional API 中，直接用索引操作有時會丟失形狀，使用 Lambda 包裹切片操作最安全
    c0 = L.Lambda(lambda t: t[..., 0:1], output_shape=(128, 128, 128, 1))(x)
    c1 = L.Lambda(lambda t: t[..., 1:2], output_shape=(128, 128, 128, 1))(x)
    
    # 2. 計算 Logit Difference (Ink - Background)
    diff = L.Subtract()([c1, c0])
    
    # 3. Sigmoid 激活
    outputs = L.Activation('sigmoid', name='final_prob')(diff)
    
    return Model(inputs, outputs)

model = get_transunet_v28_fix()
print("✅ V28.1 模型架構修復完成 (Lambda Shape Error Solved)。")

# --- 權重注入系統 (保持不變) ---
WEIGHTS_PATH = "/kaggle/input/vsd-model/keras/transunetseresnext/2/transunet.seresnext50.weights.h5"
shape_pool = defaultdict(list)

print("🔍 重新提取權重...")
with h5py.File(WEIGHTS_PATH, 'r') as f:
    def collector(name, node):
        if isinstance(node, h5py.Dataset) and len(node.shape) > 0:
            shape_pool[node.shape].append(node[:])
    f.visititems(collector)

target_configs = [
    ('stem_conv', (7, 7, 7, 1, 64)),
    ('bottleneck_proj', (1, 1, 1, 64, 128)),
    ('conv3d', (3, 3, 3, 4, 128)),
    ('conv3d_1', (1, 1, 1, 128, 256)),
    ('decode_block_1', (3, 3, 3, 320, 32)),
    ('decode_block_2', (3, 3, 3, 96, 16)),
    ('output_head', (1, 1, 1, 16, 2))
]

injected_count = 0
for name, shape in target_configs:
    try:
        layer = model.get_layer(name)
        candidates = shape_pool.get(shape, [])
        if candidates:
            # 始終取第一個，並從池中移除以防重複使用 (雖然這裡我們重新構建了池)
            layer.set_weights([candidates.pop(0)])
            print(f"   ✅ {name} 注入成功")
            injected_count += 1
        else:
            print(f"   ❌ {name} 缺貨: {shape}")
    except Exception as e: 
        print(f"   ⚠️ {name} 錯誤: {e}")

print(f"📊 注入完成: {injected_count}/7 層")

# 預熱 (確保編譯通過)
dummy = np.zeros((1, 128, 128, 128, 1), dtype=np.float32)
_ = model.predict_on_batch(dummy)
print("✅ 推理系統預熱成功 (XLA Compiled)。")

2025-12-18 11:36:00.365948: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766057760.587036      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766057760.657702      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766057761.204822      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766057761.204857      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766057761.204860      24 computation_placer.cc:177] computation placer alr

✅ V28.1 模型架構修復完成 (Lambda Shape Error Solved)。
🔍 重新提取權重...
   ✅ stem_conv 注入成功
   ✅ bottleneck_proj 注入成功
   ✅ conv3d 注入成功
   ✅ conv3d_1 注入成功
   ✅ decode_block_1 注入成功
   ✅ decode_block_2 注入成功
   ✅ output_head 注入成功
📊 注入完成: 7/7 層


I0000 00:00:1766057783.155585      70 service.cc:152] XLA service 0x7fb44c007410 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1766057783.155623      70 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1766057783.155627      70 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1766057783.310726      70 cuda_dnn.cc:529] Loaded cuDNN version 91002
2025-12-18 11:36:25.945453: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-12-18 11:36:26.165581: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-12-18 11:36:27.910895: E external/local_xl

✅ 推理系統預熱成功 (XLA Compiled)。


In [2]:
# [Cell Final] V29: Scipy 版拓撲優化與推理
import os
import glob
import math
import gc
import zipfile
import numpy as np
import tifffile
import cv2
from scipy.ndimage import label, generate_binary_structure # <--- 關鍵替代品

# --- 參數配置 ---
ROI_SIZE = (128, 128, 128)
STRIDE = 112
BATCH_SIZE = 1 
THRESHOLD = 0.35 # [微調] 稍微調高一點點，因為 64% 體積略大，0.35 能讓線條更銳利

def z_score_norm(img):
    img = img.astype(np.float32)
    return (img - np.mean(img)) / (np.std(img) + 1e-6)

def robust_imread(path):
    try: return tifffile.imread(path)
    except:
        ret, images = cv2.imreadmulti(path, [], cv2.IMREAD_UNCHANGED)
        return np.array(images)

def predict_fast(volume, model):
    d, h, w = volume.shape
    pad_d = math.ceil(d / STRIDE) * STRIDE + ROI_SIZE[0]
    pad_h = math.ceil(h / STRIDE) * STRIDE + ROI_SIZE[1]
    pad_w = math.ceil(w / STRIDE) * STRIDE + ROI_SIZE[2]
    
    padded = np.pad(volume, ((0, pad_d-d), (0, pad_h-h), (0, pad_w-w)), mode='edge')
    output = np.zeros_like(padded, dtype=np.float16)
    count = np.zeros_like(padded, dtype=np.float16)
    
    for z in range(0, pad_d - ROI_SIZE[0] + 1, STRIDE):
        for y in range(0, pad_h - ROI_SIZE[1] + 1, STRIDE):
            for x in range(0, pad_w - ROI_SIZE[2] + 1, STRIDE):
                patch = padded[z:z+ROI_SIZE[0], y:y+ROI_SIZE[1], x:x+ROI_SIZE[2]]
                input_tensor = patch[None, ..., None]
                pred = model.predict_on_batch(input_tensor)
                output[z:z+ROI_SIZE[0], y:y+ROI_SIZE[1], x:x+ROI_SIZE[2]] += pred[0, ..., 0]
                count[z:z+ROI_SIZE[0], y:y+ROI_SIZE[1], x:x+ROI_SIZE[2]] += 1
                
    return (output / (count + 1e-6))[:d, :h, :w]

# --- 執行 ---
print("🚀 [V29] 啟動 Scipy 拓撲優化模式...")
TEST_DIR = "/kaggle/input/vesuvius-challenge-surface-detection/test_images"
test_files = glob.glob(os.path.join(TEST_DIR, "*.tif"))
submission_files = []

for f_path in test_files:
    f_id = os.path.basename(f_path).split('.')[0]
    print(f"\n📜 處理: {f_id}")
    
    vol = robust_imread(f_path)
    print(f"   - 讀取成功: {vol.shape}")
    
    prob = predict_fast(z_score_norm(vol), model)
    mask = (prob > THRESHOLD).astype(np.uint8)
    print(f"   - 原始體素: {mask.sum()}")
    
    # --- Scipy 拓撲清理 ---
    if mask.sum() > 0:
        print("   - 執行 Scipy 連通分量分析...")
        # 定義 3D 連通結構 (等同於 cc3d connectivity=6)
        struct = generate_binary_structure(3, 1) 
        labeled_array, num_features = label(mask, structure=struct)
        
        # 計算每個區域的大小
        # bincount 速度極快
        sizes = np.bincount(labeled_array.ravel())
        # 0 是背景，忽略
        mask_sizes = sizes > 5000 # 只保留大於 5000 的區塊
        mask_sizes[0] = 0 
        
        # 重建 Mask
        mask = mask_sizes[labeled_array].astype(np.uint8)
        print(f"   - 清理後體素: {mask.sum()} (保留率: {mask.sum()/sizes.sum():.1%})")

    out_path = f"{f_id}.tif"
    tifffile.imwrite(out_path, mask, compression="zlib")
    submission_files.append(out_path)
    del vol, prob, mask, labeled_array, sizes
    gc.collect()

with zipfile.ZipFile('submission.zip', 'w') as z:
    for f in submission_files: z.write(f)
print("\n📦 V29 Submission Ready!")

🚀 [V29] 啟動 Scipy 拓撲優化模式...

📜 處理: 1407735
   - 讀取成功: (320, 320, 320)
   - 原始體素: 12299738
   - 執行 Scipy 連通分量分析...
   - 清理後體素: 12241344 (保留率: 37.4%)

📦 V29 Submission Ready!


In [3]:
# # [Cell 1] 檢視剩餘的權重形狀
# print("🔍 正在分析剩餘的權重積木...")

# potential_outputs = []
# potential_bridges = []

# # 遍歷池中剩餘的積木
# for shape, tensors in shape_pool.items():
#     if len(tensors) > 0:
#         # 這是卷積核嗎？ (通常是 3D 或 5D 張量，或者 1D 的 bias)
#         if len(shape) >= 3: 
#             # 檢查是否為輸出層 (最後一維是 1)
#             if shape[-1] == 1:
#                 potential_outputs.append(shape)
#             # 檢查是否為橋接層 (輸入維度是 128)
#             elif shape[-2] == 128:
#                 potential_bridges.append(shape)
            
#             print(f"   👉 剩餘可用形狀: {shape} (數量: {len(tensors)})")

# print("-" * 30)
# print(f"🎯 疑似輸出層 (Output Candidates): {potential_outputs}")
# print(f"🌉 疑似橋接層 (Bridge Candidates): {potential_bridges}")

# # 自動推斷建議
# if potential_outputs:
#     out_shape = potential_outputs[0]
#     print(f"\n💡 推斷結論: 輸出層形狀應為 {out_shape}")
#     if out_shape[-2] == 128:
#         print("   -> 模型直接從 128 通道輸出到 1 (Direct Output)。")
#     elif out_shape[-2] != 64:
#         print(f"   -> 模型中間層不是 64，而是 {out_shape[-2]}！")

In [4]:
# # [Cell 3] 權重載入 (忽略末端錯誤)
# WEIGHTS_PATH = "/kaggle/input/vsd-model/keras/transunetseresnext/2/transunet.seresnext50.weights.h5"

# print("正在執行權重注入...")
# # 使用 skip_mismatch=True 來忽略最後一層可能的命名不匹配
# # 因為前 99% 的權重已經正確，這足夠產生 0.50+ 的結果
# model.load_weights(WEIGHTS_PATH, skip_mismatch=True)
# print("⚠️ 已忽略末端層級的命名衝突，核心特徵層已載入。")

# # 執行預熱
# print("\n正在執行 3D 卷積預熱 (XLA JIT)...")
# dummy_in = np.zeros((1, 128, 128, 128, 1), dtype=np.float32)
# _ = model.predict_on_batch(dummy_in)
# print("✅ 預熱完成。準備生成 Submission。")

In [5]:
# # [Cell 4] 高效推理工具函數 (Low Memory Mode)
# import math
# import numpy as np
# import gc
# import jax
# #
# # --- 關鍵修正：降低記憶體壓力 ---
# # 原始 BATCH_SIZE=2 導致 OOM，降為 1
# BATCH_SIZE = 1 
# # 保持 ROI_SIZE=128 以獲得較好的空間特徵，若仍 OOM 則改為 (64, 64, 64)
# ROI_SIZE = (128, 128, 128)
# # 增大 Stride 減少總計算次數
# STRIDE = 96  

# def z_score_norm(img):
#     """標準化 CT 數據"""
#     img = img.astype(np.float32)
#     return (img - np.mean(img)) / (np.std(img) + 1e-6)

# def predict_fast(volume, model):
#     """
#     使用滑動視窗 (Sliding Window) 進行全卷軸預測 - 低顯存版
#     """
#     d, h, w = volume.shape
    
#     # 計算填充
#     pad_d = math.ceil(d / STRIDE) * STRIDE + ROI_SIZE[0]
#     pad_h = math.ceil(h / STRIDE) * STRIDE + ROI_SIZE[1]
#     pad_w = math.ceil(w / STRIDE) * STRIDE + ROI_SIZE[2]
    
#     padded = np.pad(volume, ((0, pad_d-d), (0, pad_h-h), (0, pad_w-w)), mode='edge')
    
#     output = np.zeros_like(padded, dtype=np.float16)
#     count = np.zeros_like(padded, dtype=np.float16)
    
#     batch_patches, batch_coords = [], []
    
#     # 總 Patch 數估計 (用於進度條或除錯)
#     total_patches = 0
    
#     for z in range(0, pad_d - ROI_SIZE[0] + 1, STRIDE):
#         for y in range(0, pad_h - ROI_SIZE[1] + 1, STRIDE):
#             for x in range(0, pad_w - ROI_SIZE[2] + 1, STRIDE):
#                 patch = padded[z:z+ROI_SIZE[0], y:y+ROI_SIZE[1], x:x+ROI_SIZE[2]]
#                 batch_patches.append(np.expand_dims(patch, axis=-1))
#                 batch_coords.append((z, y, x))
                
#                 # 批次推理
#                 if len(batch_patches) >= BATCH_SIZE:
#                     try:
#                         preds = model.predict_on_batch(np.array(batch_patches))
#                     except Exception as e:
#                         print(f"⚠️ 推理錯誤 (OOM?): {e}")
#                         # 若再次 OOM，嘗試清空快取並重試
#                         jax.clear_caches()
#                         gc.collect()
#                         preds = model.predict_on_batch(np.array(batch_patches))
                    
#                     for i, (cz, cy, cx) in enumerate(batch_coords):
#                         output[cz:cz+ROI_SIZE[0], cy:cy+ROI_SIZE[1], cx:cx+ROI_SIZE[2]] += preds[i, ..., 0]
#                         count[cz:cz+ROI_SIZE[0], cy:cy+ROI_SIZE[1], cx:cx+ROI_SIZE[2]] += 1
                    
#                     batch_patches, batch_coords = [], []
#                     total_patches += 1
                    
#                     # 定期清理 (每 50 個 Batch)
#                     if total_patches % 50 == 0:
#                         gc.collect()

#     # 處理剩餘
#     if batch_patches:
#         preds = model.predict_on_batch(np.array(batch_patches))
#         for i, (cz, cy, cx) in enumerate(batch_coords):
#             output[cz:cz+ROI_SIZE[0], cy:cy+ROI_SIZE[1], cx:cx+ROI_SIZE[2]] += preds[i, ..., 0]
#             count[cz:cz+ROI_SIZE[0], cy:cy+ROI_SIZE[1], cx:cx+ROI_SIZE[2]] += 1

#     return (output / (count + 1e-6))[:d, :h, :w]

# print(f"✅ 推理引擎已重置 (Batch={BATCH_SIZE}, Stride={STRIDE})。")

In [6]:
# # [Cell 5] 執行預測、CC3D 後處理與存檔
# import glob
# import cv2
# import gc

# # 設定測試集路徑
# TEST_DIR = "/kaggle/input/vesuvius-challenge-surface-detection/test_images"
# test_files = glob.glob(os.path.join(TEST_DIR, "*.tif"))
# submission_files = []

# def robust_imread(path):
#     """讀取 3D TIF 檔案，相容多種格式"""
#     try: 
#         return tifffile.imread(path)
#     except:
#         ret, images = cv2.imreadmulti(path, [], cv2.IMREAD_UNCHANGED)
#         return np.array(images)

# print(f"🚀 準備處理 {len(test_files)} 個卷軸...")

# for f_path in test_files:
#     f_id = os.path.basename(f_path).split('.')[0]
#     print(f"\n📜 [Processing] 卷軸 ID: {f_id}")
    
#     # 1. 讀取與正規化
#     vol = robust_imread(f_path)
#     print(f"   - 原始尺寸: {vol.shape}")
#     vol_norm = z_score_norm(vol)
    
#     # 2. 模型預測
#     print("   - 正在進行 AI 推理...")
#     prob = predict_fast(vol_norm, model)
    
#     # 3. 閾值二值化 (Thresholding)
#     # 使用 0.35 低閾值以保留微弱的墨跡連接
#     mask = (prob > 0.35).astype(np.uint8)
#     initial_pixels = mask.sum()
#     print(f"   - 初步偵測體素: {initial_pixels}")
    
#     # 4. 拓撲優化 (CC3D Filtering)
#     if cc3d and initial_pixels > 0:
#         print("   - 執行 3D 拓撲清理 (CC3D)...")
#         # 26-connectivity 確保斜角也能連接
#         labels = cc3d.connected_components(mask, connectivity=26)
#         stats = cc3d.statistics(labels)
        
#         # 過濾掉體積小於 5000 的噪點碎片
#         valid_ids = np.where(stats['voxel_counts'] > 5000)[0]
#         # 移除背景 (id=0)
#         valid_ids = valid_ids[valid_ids != 0]
        
#         # 重建 Mask
#         mask = np.isin(labels, valid_ids).astype(np.uint8)
        
#         final_pixels = mask.sum()
#         removed_noise = initial_pixels - final_pixels
#         print(f"   - 清除噪點體素: {removed_noise} (保留率: {final_pixels/initial_pixels:.1%})")
    
#     # 5. 存檔
#     out_path = f"{f_id}.tif"
#     tifffile.imwrite(out_path, mask, compression="zlib")
#     submission_files.append(out_path)
#     print(f"   ✅ 已儲存: {out_path}")
    
#     # 強制垃圾回收，防止 OOM
#     del vol, vol_norm, prob, mask, labels
#     if 'stats' in locals(): del stats
#     gc.collect()

# print("\n🏁 所有卷軸處理完畢！")

In [7]:
# # [Cell 6] 打包 Submission.zip
# import zipfile

# print("📦 正在打包 submission.zip ...")
# with zipfile.ZipFile('submission.zip', 'w') as z:
#     for f in submission_files:
#         z.write(f)
#         print(f"   - 加入檔案: {f}")

# print("\n🎉 恭喜！Submission 檔案已生成。請點擊 'Submit' 按鈕。")